In [0]:
# Teste de ambiente PySpark no Databricks
from pyspark.sql import functions as F

data = [("Porto do Rio de Janeiro", 12.5), ("Porto de Santos", 14.2), ("Porto de Paranaguá", 13.0)]
columns = ["nome_porto", "calado_maximo_m"]

df_teste = spark.createDataFrame(data, schema=columns)
display(df_teste.withColumn("status_operacional", F.when(F.col("calado_maximo_m") >= 13.0, "Profundo").otherwise("Restrito")))

In [0]:
# ==============================================================================
# PASSO 1 (CORRIGIDO): INGESTÃO E CRIAÇÃO DA CAMADA BRONZE (RAW DATA)
# ==============================================================================

import pandas as pd
from pyspark.sql import functions as F

# 1. Criar o Banco de Dados para a Camada Bronze no Catálogo
spark.sql("CREATE DATABASE IF NOT EXISTS mvp_s3_bronze")

# 2. URL do arquivo Raw no seu GitHub
url_dados_antaq = "https://raw.githubusercontent.com/ThiagoFOPinto/puc-data-science-analytics/main/mvp_s2_predictive/data/puc_infra_portos.csv"

try:
    print("Baixando e lendo o arquivo do GitHub via Pandas...")
    
    # Lemos a URL HTTP diretamente para a memória via Pandas
    df_pd = pd.read_csv(url_dados_antaq)
    
    # Padronizar nomes das colunas (em minúsculas e sem espaços)
    df_pd.columns = [c.strip().lower().replace(" ", "_") for c in df_pd.columns]
    
    # Converter para DataFrame do PySpark
    df_bronze_raw = spark.createDataFrame(df_pd)
    
    print("✅ Base do GitHub baixada e convertida para PySpark com sucesso!")

except Exception as e:
    print(f"Erro ao carregar dados: {e}")

# 3. Adicionar Metadados de Ingestão e Governança
df_bronze = df_bronze_raw \
    .withColumn("dh_ingestao", F.current_timestamp()) \
    .withColumn("fonte_dados", F.lit("GitHub_puc_infra_portos"))

# 4. Salvar na Camada Bronze (Tabela Delta)
df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("mvp_s3_bronze.infra_portos")

print("✅ Ingestão da Camada Bronze concluída com sucesso!")
print(f"Total de registros salvos na tabela Bronze: {spark.table('mvp_s3_bronze.infra_portos').count()}")

# Exibir prévia dos dados salvos no catálogo
display(spark.table("mvp_s3_bronze.infra_portos").limit(5))

In [0]:
# ==============================================================================
# PASSO 2: CAMADA SILVER - LIMPEZA, TIPAGEM E QUALIDADE DE DADOS
# ==============================================================================

from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType

# 1. Criar o Banco de Dados para a Camada Silver
spark.sql("CREATE DATABASE IF NOT EXISTS mvp_s3_silver")

# 2. Ler os dados brutos da Camada Bronze
df_bronze_input = spark.table("mvp_s3_bronze.infra_portos")

print(f"📊 Registros na Camada Bronze (Entrada): {df_bronze_input.count()}")

# 3. Tratamentos de Qualidade e Padronização na Silver
df_silver_clean = (
    df_bronze_input
    # A. Unicidade: Remoção de registros duplicados
    .dropDuplicates()
    
    # B. Padronização de Texto (Porto com inicial maiúscula e sem espaços extras)
    .withColumn("porto", F.initcap(F.trim(F.col("porto"))))
    
    # C. Tipagem Adequada (Conversão de métricas numéricas para Double/Integer)
    .withColumn("dwt_medio", F.col("dwt_medio").cast(DoubleType()))
    .withColumn("ano", F.col("ano").cast(IntegerType()))
    .withColumn("mes", F.col("mes").cast(IntegerType()))
    
    # D. Validação de Acurácia: Garantir DWT válido (positivo ou nulo)
    .filter((F.col("dwt_medio").isNull()) | (F.col("dwt_medio") >= 0))
    
    # E. Metadados da Silver
    .withColumn("dh_processamento_silver", F.current_timestamp())
)

# 4. Salvar persistentemente na Camada Silver (Tabela Delta)
df_silver_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("mvp_s3_silver.infra_portos")

print("✅ Camada Silver processada e salva com sucesso!")
print(f"📊 Registros validados na Camada Silver: {spark.table('mvp_s3_silver.infra_portos').count()}")

# Exibir prévia dos dados limpos
display(spark.table("mvp_s3_silver.infra_portos").limit(5))

In [0]:
# ==============================================================================
# PASSO 3: CAMADA GOLD - MODELAGEM DIMENSIONAL (STAR SCHEMA)
# ==============================================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Criar o Banco de Dados para a Camada Gold
spark.sql("CREATE DATABASE IF NOT EXISTS mvp_s3_gold")

# 2. Carregar dados validados da Camada Silver
df_silver = spark.table("mvp_s3_silver.infra_portos")

# ------------------------------------------------------------------------------
# A. CRIAR A DIMENSÃO PORTO (dim_porto)
# ------------------------------------------------------------------------------
window_porto = Window.orderBy("porto")

df_dim_porto = (
    df_silver.select("porto")
    .distinct()
    .filter(F.col("porto").isNotNull())
    .withColumn("id_porto", F.row_number().over(window_porto))
    .select("id_porto", "porto")
)

df_dim_porto.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("mvp_s3_gold.dim_porto")
print(f"✅ Dimensão Porto criada: {spark.table('mvp_s3_gold.dim_porto').count()} registros.")

# ------------------------------------------------------------------------------
# B. CRIAR A DIMENSÃO TEMPO (dim_tempo)
# ------------------------------------------------------------------------------
window_tempo = Window.orderBy("ano", "mes")

df_dim_tempo = (
    df_silver.select("ano", "mes")
    .distinct()
    .filter(F.col("ano").isNotNull() & F.col("mes").isNotNull())
    .withColumn("trimestre", F.ceil(F.col("mes") / 3))
    .withColumn("semestre", F.ceil(F.col("mes") / 6))
    .withColumn("id_tempo", F.row_number().over(window_tempo))
    .select("id_tempo", "ano", "mes", "trimestre", "semestre")
)

df_dim_tempo.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("mvp_s3_gold.dim_tempo")
print(f"✅ Dimensão Tempo criada: {spark.table('mvp_s3_gold.dim_tempo').count()} registros.")

# ------------------------------------------------------------------------------
# C. CRIAR A TABELA FATO INFRAESTRUTURA (fato_infra_operacional)
# ------------------------------------------------------------------------------
df_fato = (
    df_silver
    .join(df_dim_porto, on="porto", how="inner")
    .join(df_dim_tempo, on=["ano", "mes"], how="inner")
    .select(
        "id_porto",
        "id_tempo",
        "dwt_medio",
        F.current_timestamp().alias("dh_processamento_gold")
    )
)

df_fato.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("mvp_s3_gold.fato_infra_operacional")
print(f"✅ Tabela Fato criada: {spark.table('mvp_s3_gold.fato_infra_operacional').count()} registros.")

# ------------------------------------------------------------------------------
# D. PRÉVIA DA MODELAGEM NA CAMADA GOLD
# ------------------------------------------------------------------------------
print("\n🔍 Prévia da Tabela Fato unida às Dimensões:")
df_gold_view = (
    spark.table("mvp_s3_gold.fato_infra_operacional")
    .join(spark.table("mvp_s3_gold.dim_porto"), "id_porto")
    .join(spark.table("mvp_s3_gold.dim_tempo"), "id_tempo")
    .select("porto", "ano", "mes", "trimestre", "dwt_medio")
)

display(df_gold_view.limit(5))

In [0]:
# ==============================================================================
# PASSO 4: CONSULTAS ANALÍTICAS NAS TABELAS DA CAMADA GOLD (SPARK SQL)
# ==============================================================================

# Registrar a visão integrada da Gold para consultas SQL limpas
spark.sql("""
CREATE OR REPLACE TEMP VIEW vw_gold_analytics AS
SELECT 
    p.porto,
    t.ano,
    t.mes,
    t.trimestre,
    t.semestre,
    f.dwt_medio
FROM mvp_s3_gold.fato_infra_operacional f
JOIN mvp_s3_gold.dim_porto p ON f.id_porto = p.id_porto
JOIN mvp_s3_gold.dim_tempo t ON f.id_tempo = t.id_tempo
""")

print("📊 PERGUNTA 1: Capacidade Média de Porte (DWT) por Porto")
display(spark.sql("""
    SELECT 
        porto,
        ROUND(AVG(dwt_medio), 2) AS dwt_medio_geral,
        ROUND(MAX(dwt_medio), 2) AS dwt_maximo,
        COUNT(*) AS total_registros
    FROM vw_gold_analytics
    GROUP BY porto
    ORDER BY dwt_medio_geral DESC
"""))

print("📊 PERGUNTA 2: Evolução Anual do DWT Médio por Complexo Portuário")
display(spark.sql("""
    SELECT 
        ano,
        porto,
        ROUND(AVG(dwt_medio), 2) AS dwt_medio_ano
    FROM vw_gold_analytics
    GROUP BY ano, porto
    ORDER BY ano ASC, dwt_medio_ano DESC
"""))

print("📊 PERGUNTA 3: Distribuição Trimestral da Capacidade Operacional")
display(spark.sql("""
    SELECT 
        trimestre,
        ROUND(AVG(dwt_medio), 2) AS dwt_medio_trimestral
    FROM vw_gold_analytics
    GROUP BY trimestre
    ORDER BY trimestre ASC
"""))